## Automatic annotation using INSID3 Model 

### before you use this scripts, you need create the reference image and mask .

### Just like assets/example ref_* source, one example img and its mask .

In [3]:
## generate mask
import json
import sys
import numpy as np
from pathlib import Path
from PIL import Image, ImageDraw
# Ensure project root is on sys.path so `pkg` can be imported
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "src":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

def generate_mask_from_annotation(reference_image_path, annotation_path, mask_output_path):
    """
    Generate a binary mask PNG from an annotation file (YOLO .txt or COCO .json).

    YOLO format (.txt): one line per object → class_id x1 y1 x2 y2 ... (normalized 0-1)
    COCO format (.json): standard COCO JSON with polygon segmentation
    """
    ref_img = Image.open(reference_image_path).convert("RGB")
    w, h = ref_img.size
    mask = Image.new("L", (w, h), 0)
    draw = ImageDraw.Draw(mask)

    annotation_path = Path(annotation_path)
    suffix = annotation_path.suffix.lower()

    if suffix == ".txt":
        with open(annotation_path, "r") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                parts = line.split()
                if len(parts) < 7:
                    print(f"  ⚠ Skipping line (need ≥7 values for polygon): {line[:60]}...")
                    continue
                class_id = parts[0]
                coords = list(map(float, parts[1:]))
                if len(coords) % 2 != 0:
                    print(f"  ⚠ Skipping line (odd coords): {line[:60]}...")
                    continue
                points = []
                for i in range(0, len(coords), 2):
                    px = coords[i] * w
                    py = coords[i + 1] * h
                    points.append((px, py))
                draw.polygon(points, fill=255)

    elif suffix == ".json":
        with open(annotation_path, "r") as f:
            coco = json.load(f)
        ref_name = Path(reference_image_path).name
        image_id = None
        for img_info in coco.get("images", []):
            if img_info["file_name"] == ref_name:
                image_id = img_info["id"]
                break
        if image_id is None:
            raise ValueError(f"Image '{ref_name}' not found in COCO annotations")
        for ann in coco.get("annotations", []):
            if ann["image_id"] != image_id:
                continue
            for seg in ann.get("segmentation", []):
                points = [(seg[i], seg[i + 1]) for i in range(0, len(seg), 2)]
                draw.polygon(points, fill=255)

    else:
        raise ValueError(f"Unsupported annotation format: {suffix}")

    mask_output_path = Path(mask_output_path)
    mask_output_path.parent.mkdir(parents=True, exist_ok=True)
    mask.save(mask_output_path)
    print(f"✅ Mask saved to {mask_output_path}  ({w}×{h})")





In [4]:
reference_image = PROJECT_ROOT / "assets/example/ref_giftbox_image.jpg"
annotation_file = PROJECT_ROOT / "assets/example/ref_giftbox_image.txt"    # yolo or coco .json
mask_output = PROJECT_ROOT / "assets/example/ref_giftbox_mask.png"

generate_mask_from_annotation(reference_image, annotation_file, mask_output)

✅ Mask saved to /home/yy/yy/github/automatic-annotation/assets/example/ref_giftbox_mask.png  (640×480)


In [ ]:

from pathlib import Path

import numpy as np
from skimage import measure



from pkg.insid3.models import build_insid3

# Build model
model = build_insid3()


ref_image_path = PROJECT_ROOT / "assets/example/ref_giftbox_image.jpg"
ref_mask_path = PROJECT_ROOT / "assets/example/ref_giftbox_mask.png"
model.set_reference(str(ref_image_path), str(ref_mask_path))


target_folder = PROJECT_ROOT / "assets/giftbox_task/images"
annotation_save_path = PROJECT_ROOT / "assets/giftbox_task/annotations/annotations.json"


target_paths = sorted(target_folder.glob("*"))
print(f"Found {len(target_paths)} target images")


In [ ]:
import json
from PIL import Image


def make_coco_annotation_from_insid3(
    model,
    target_paths,
    save_path,
    class_name="object",
    min_area=0
):
    """
    Run INSID3 segment on each target image and convert to COCO format annotation.json

    Args:
        model: INSID3 model (reference already set)
        target_paths: list of Path to target images
        save_path: Path to save the annotation.json
        class_name: category name (single-class)
        min_area: minimum mask area to keep (filters noise)
    """
    category_id = 1

    coco_output = {
        "images": [],
        "annotations": [],
        "categories": [
            {"id": category_id, "name": class_name, "supercategory": "object"}
        ],
    }

    ann_id = 1

    for img_id, target_path in enumerate(target_paths, start=1):
        # Read image to get dimensions
        target_img = Image.open(target_path)
        width, height = target_img.size

        coco_output["images"].append({
            "id": img_id,
            "file_name": target_path.name,
            "width": width,
            "height": height,
        })

        # Run INSID3 segmentation
        model.set_target(str(target_path))
        pred_mask = model.segment()  # (H, W) bool tensor

        if isinstance(pred_mask, np.ndarray):
            mask_np = pred_mask
        else:
            mask_np = pred_mask.cpu().numpy()

        if not mask_np.any():
            print(f"  ⚠ No mask found for {target_path.name}, skipping...")
            continue

        # Find connected components
        labeled = measure.label(mask_np.astype(np.uint8), connectivity=2)
        regions = measure.regionprops(labeled)

        for region in regions:
            if region.area < min_area:
                continue

            # Extract polygon from binary mask
            region_mask = (labeled == region.label)
            contours = measure.find_contours(region_mask.astype(np.float64), 0.5)

            if not contours:
                continue

            # Use the largest contour
            contour = max(contours, key=len)
            # Convert (row, col) -> (x, y)
            contour_xy = np.fliplr(contour)
            segmentation = contour_xy.flatten().tolist()

            # Bbox: [x, y, width, height]
            y1, x1, y2, x2 = region.bbox
            bbox_w = x2 - x1
            bbox_h = y2 - y1

            coco_output["annotations"].append({
                "id": ann_id,
                "image_id": img_id,
                "category_id": category_id,
                "segmentation": [segmentation],
                "bbox": [float(x1), float(y1), float(bbox_w), float(bbox_h)],
                "area": float(region.area),
                "iscrowd": 0
            })
            ann_id += 1

        if img_id % 10 == 0:
            print(f"  Processed {img_id}/{len(target_paths)} images...")

    # Save
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(coco_output, f, indent=2, ensure_ascii=False)

    # Save labels
    labels_path = save_path.parent / "labels.txt"
    with open(labels_path, "w", encoding="utf-8") as f_labels:
        f_labels.write(f"{class_name}\n")

    num_imgs = len(coco_output["images"])
    num_anns = len(coco_output["annotations"])
    num_cats = len(coco_output["categories"])
    print(f"✅ COCO annotation saved to {save_path}")
    print(f"   images={num_imgs}, annotations={num_anns}, categories={num_cats}")


In [ ]:
# Run inference on all target images and save annotations
if not annotation_save_path.exists():
    annotation_save_path.parent.mkdir(parents=True, exist_ok=True)

make_coco_annotation_from_insid3(
    model=model,
    target_paths=target_paths,
    save_path=annotation_save_path,
    class_name="object"
)
